In [4]:
# Install necessary Python libraries for data manipulation, machine learning, and LLM integration.
!pip install pandas numpy scikit-learn imbalanced-learn shap langchain langchain-anthropic langchain-openai langchain-community langchain-ollama langgraph matplotlib seaborn requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.0/49.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/635.9 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28
ERROR: pip's dependenc

In [6]:
# Upload the dataset files (KDDTrain+.txt and KDDTest+.txt) from your local machine to the Colab environment.
from google.colab import files
uploaded = files.upload()

Saving KDDTest+.txt to KDDTest+.txt


In [7]:
import pandas as pd

# Define the column names for the NSL-KDD dataset.
# This dataset contains 41 features, a 'label' indicating the type of connection,
# and a 'difficulty' score.
col_names = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes',
    'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root',
    'num_file_creations','num_shells','num_access_files',
    'num_outbound_cmds','is_host_login','is_guest_login','count',
    'srv_count','serror_rate','srv_serror_rate','rerror_rate',
    'srv_rerror_rate','same_srv_rate','diff_srv_rate',
    'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
    'dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty'
]

# Load the training and testing datasets into pandas DataFrames.
# The 'names' parameter assigns the defined column names to the data.
train_df = pd.read_csv('KDDTrain+.txt', names=col_names)
test_df  = pd.read_csv('KDDTest+.txt',  names=col_names)

# Define a mapping from the fine-grained attack labels to broader attack families.
# This simplifies the classification problem into 5 main categories: normal, DoS,
# Probe, R2L (Remote to Local), and U2R (User to Root).
# Note: If running the code as separate files, run ingest.py first before running preprocess.py
attack_map = {
    'normal': 'normal',
    'back':'dos', 'land':'dos', 'neptune':'dos', 'pod':'dos',
    'smurf':'dos', 'teardrop':'dos',
    'ipsweep':'probe', 'nmap':'probe', 'portsweep':'probe', 'satan':'probe',
    'ftp_write':'r2l', 'guess_passwd':'r2l', 'imap':'r2l',
    'multihop':'r2l', 'phf':'r2l', 'spy':'r2l',
    'warezclient':'r2l', 'warezmaster':'r2l',
    'buffer_overflow':'u2r', 'loadmodule':'u2r',
    'perl':'u2r', 'rootkit':'u2r'
}
# Apply the mapping to the 'label' column in the training DataFrame to create a new 'attack_family' column.
# .fillna('unknown') handles any labels not present in the map.
train_df['attack_family'] = train_df['label'].map(attack_map).fillna('unknown')

In [8]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from imblearn.over_sampling import SMOTE

# Step 1: One-Hot Encode categorical features to convert them into a numerical format
# that machine learning models can understand. This is done robustly.
cat_features = ['protocol_type', 'service', 'flag']

# Initialize OneHotEncoder. `handle_unknown='ignore'` is crucial to prevent errors
# if the model encounters a new categorical value during testing that wasn't in training.
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit the encoder on the training data and transform both training and test data.
train_encoded = ohe.fit_transform(train_df[cat_features])
test_encoded = ohe.transform(test_df[cat_features])
# Get the new column names generated by one-hot encoding.
ohe_cols = ohe.get_feature_names_out(cat_features)

# Combine the original DataFrame (excluding the categorical features) with the new encoded features.
train_df = pd.concat([train_df.drop(columns=cat_features),
                      pd.DataFrame(train_encoded, columns=ohe_cols, index=train_df.index)], axis=1)
test_df = pd.concat([test_df.drop(columns=cat_features),
                     pd.DataFrame(test_encoded, columns=ohe_cols, index=test_df.index)], axis=1)

# Map attack families to integer labels for test_df as well, similar to how it was done for train_df.
test_df['attack_family'] = test_df['label'].map(attack_map).fillna('unknown')

# Step 2: Prepare the feature matrices (X) and target vectors (y).
# Drop non-feature columns like original labels, difficulty score, and the attack family (which is our target).
drop_cols = ['label', 'difficulty', 'attack_family']
X_train = train_df.drop(columns=drop_cols)
X_test  = test_df.drop(columns=drop_cols)

# Create a numerical mapping for the attack families for model training.
label_map = {'normal':0, 'dos':1, 'probe':2, 'r2l':3, 'u2r':4, 'unknown':5}
y_train = train_df['attack_family'].map(label_map)
y_test  = test_df['attack_family'].map(label_map)

# Step 3: Scale numerical features. Standardization (StandardScaler) transforms data
# to have a mean of 0 and standard deviation of 1. This is important for many ML models.
# The scaler is fitted ONLY on the training data to prevent data leakage,
# then used to transform both training and test sets.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Step 4: Address class imbalance in the training set using SMOTE (Synthetic Minority Over-sampling Technique).
# This creates synthetic samples for minority classes to balance the dataset,
# which helps prevent models from being biased towards the majority class.
# `k_neighbors=3` means 3 nearest neighbors are used to generate synthetic samples.
smote = SMOTE(random_state=42, k_neighbors=3)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)

print(f'After SMOTE training samples: {len(X_train_bal)}')


After SMOTE training samples: 336715


In [10]:
from sklearn.ensemble import RandomForestClassifier, IsolationForest
import numpy as np

# Initialize and train a RandomForestClassifier, a powerful ensemble learning method.
# n_estimators=200: Uses 200 decision trees.
# max_depth=25: Limits the depth of each tree to prevent overfitting.
# min_samples_leaf=2: Requires at least 2 samples to be at a leaf node.
# class_weight='balanced': Automatically adjusts weights inversely proportional to class frequencies,
# helping with imbalanced datasets (even after SMOTE).
# random_state=42: Ensures reproducibility of results.
# n_jobs=-1: Uses all available CPU cores for faster training.
rf = RandomForestClassifier(
    n_estimators=200,      # more trees -> stable SHAP values
    max_depth=25,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train_bal, y_train_bal)

# Predict probabilities for each class for the test set. This gives confidence scores.
rf_probs = rf.predict_proba(X_test_scaled)  # shape: (N, 6) (N samples, 6 classes)
# Predict the most likely class label for each test sample.
rf_preds = rf.predict(X_test_scaled)
# Get the confidence of the predicted class (the highest probability).
rf_conf  = rf_probs.max(axis=1)


In [11]:
# Create a mask to select only 'normal' connection samples from the training data.
# This is because Isolation Forest is an unsupervised anomaly detection algorithm
# that learns from normal data to identify deviations.
normal_mask = (y_train == 0)
X_normal = X_train_scaled[normal_mask]

# Initialize and train an Isolation Forest model for anomaly detection.
# n_estimators=200: Builds 200 isolation trees.
# contamination=0.05: Specifies the expected proportion of outliers in the dataset (5%).
# random_state=42: Ensures reproducibility.
# n_jobs=-1: Uses all available CPU cores.
iso = IsolationForest(
    n_estimators=200,
    contamination=0.05,   # expect ~5% anomalies
    random_state=42,
    n_jobs=-1,
)
# Train the Isolation Forest model on only the normal training data.
iso.fit(X_normal)

# Calculate anomaly scores for the test set. More negative values indicate higher anomaly.
iso_scores = iso.decision_function(X_test_scaled)  # more negative = more anomalous
# Predict whether each test sample is an anomaly (-1) or normal (+1).
iso_preds  = iso.predict(X_test_scaled)             # -1 = anomaly, +1 = normal

In [12]:
from sklearn.preprocessing import MinMaxScaler

# Normalize the Isolation Forest scores to a [0,1] range.
# We negate the `iso_scores` because `decision_function` gives lower scores for anomalies,
# but we want higher values to indicate more anomalous behavior for easier interpretation.
# `.reshape(-1,1)` is needed because MinMaxScaler expects 2D input.
iso_norm = MinMaxScaler().fit_transform(
    (-iso_scores).reshape(-1,1)).ravel()

# Combine the predictions from both models to create a 'fused anomaly' flag.
# An anomaly is flagged if the Random Forest predicts any attack (not normal, `rf_preds != 0`)
# OR if the normalized Isolation Forest score is above a threshold (0.7).
fused_anomaly = ((rf_preds != 0) | (iso_norm > 0.7)).astype(int)

# Calculate a final 'fused confidence' score by taking a weighted average
# of the Random Forest's prediction confidence and the normalized Isolation Forest anomaly score.
# This gives more weight to the Random Forest's confidence (0.6) and less to Isolation Forest (0.4).
fused_confidence = 0.6 * rf_conf + 0.4 * iso_norm


In [13]:
import shap

# Initialize a SHAP (SHapley Additive exPlanations) explainer for the Random Forest model.
# SHAP helps explain individual predictions by showing the contribution of each feature.
explainer = shap.TreeExplainer(rf)
# Get the feature names, which are the column names of our processed training data.
feature_names = list(X_train.columns)  # Features after OHE

# Define a function to explain a single network connection prediction using SHAP values.
# It takes the original (raw) row, the scaled row, and the model's prediction.
def explain_connection(raw_row_df, row_scaled, prediction):
    # SHAP requires the scaled data to compute feature importances.
    # It calculates the SHAP values for the given scaled row.
    shap_vals = explainer.shap_values(row_scaled.reshape(1, -1))

    # Handle different SHAP output formats (list for multi-output, 3D array for multi-class with single output).
    if isinstance(shap_vals, list):
        class_shap = shap_vals[prediction][0]
    elif len(shap_vals.shape) == 3:
        class_shap = shap_vals[0, :, prediction]
    else:
        class_shap = shap_vals[0]

    # Identify the top 8 features that contribute most to the prediction (based on absolute SHAP value).
    top_idx = np.argsort(np.abs(class_shap))[::-1][:8]

    drivers = []
    for i in top_idx:
        drivers.append({
            'feature': feature_names[i],
            # IMPORTANT FIX: Retrieve the original, unscaled value of the feature
            # from the raw DataFrame (`raw_row_df`) for user-friendly interpretation.
            # In the drivers loop, for OHE features, strip the prefix to recover category
            # Note: OHE cols show 0/1 not original string
            'true_value': float(raw_row_df.iloc[0, i]),
            'shap_value': float(class_shap[i]),
            'direction': 'increases risk' if class_shap[i] > 0 else 'decreases risk',
        })

    # Define human-readable class names for the predictions.
    class_names = ['normal','dos','probe','r2l','u2r','unknown']
    # compute proba for just this row
    row_proba = rf.predict_proba(row_scaled.reshape(1, -1))[0]

    # Get the base value (expected output of the model if no features were present).
    # This also handles different formats for `explainer.expected_value`.
    if isinstance(explainer.expected_value, (list, np.ndarray)):
        base_val = float(explainer.expected_value[prediction])
    else:
        base_val = float(explainer.expected_value)

    # Return a dictionary containing the predicted class, model confidence, top SHAP drivers,
    # and the base value, forming a 'SHAP bundle' for explanation.
    return {
        'predicted_class':  class_names[prediction],
        'rf_confidence':    float(row_proba.max()),
        'top_shap_drivers': drivers,
        'base_value':       base_val
    }


In [14]:
# Install the `langchain-ollama` library, which provides integration with Ollama models.
!pip install -U langchain-ollama

# Install and run the Ollama server in Google Colab.
# `sudo apt-get update && sudo apt-get install -y zstd` installs a compression utility required by Ollama.
# `curl -fsSL https://ollama.com/install.sh | sh` downloads and executes the Ollama installation script.
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Run Ollama in the background as a separate process.
import subprocess
import os

# Check if the Ollama server is already running to avoid starting multiple instances.
try:
    # Attempt to run an Ollama command; if it succeeds, the server is likely running.
    subprocess.run(['ollama', 'ps'], capture_output=True, check=True)
    print("Ollama server is already running.")
except (subprocess.CalledProcessError, FileNotFoundError):
    print("Starting Ollama server...")
    # Start Ollama in a new process group. `preexec_fn=os.setsid` detaches it from the current process,
    # allowing Colab to continue running while Ollama runs in the background.
    process = subprocess.Popen(['ollama', 'serve'], preexec_fn=os.setsid,
                               stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    # Give the server a few seconds to fully start up.
    import time
    time.sleep(5)
    print("Ollama server started.")

# Pull (download) the 'mistral' large language model from Ollama's model library.
# `capture_output=False` shows the download progress in real-time.
print("Pulling mistral model...")
subprocess.run(['ollama', 'pull', 'mistral'], capture_output=False, check=True)
print("Mistral model pulled.")


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,970 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,931 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,623 kB]
Hit:

In [15]:
import os
import requests
import json
from langchain_core.tools import tool

# Import `create_react_agent` from `langgraph.prebuilt` to construct an agent
# that can reason and use tools.
from langgraph.prebuilt import create_react_agent

# Define a custom tool to query AbuseIPDB for IP address reputation.
# `@tool` decorator makes this Python function available to the LLM agent.
@tool
def lookup_ip_reputation(ip_address: str) -> str:
    """
    Queries the AbuseIPDB API to retrieve threat intelligence for a given IP address.
    Returns the abuse confidence score, ISP, and recent reports.
    """
    # Get the API key from environment variables for security.
    api_key = os.getenv("ABUSEIPDB_API_KEY")
    if not api_key:
        return f"System Error: ABUSEIPDB_API_KEY environment variable not set. Cannot verify {ip_address}."

    # Construct the API request details.
    url = "https://api.abuseipdb.com/api/v2/check"
    querystring = {'ipAddress': ip_address, 'maxAgeInDays': '90'}
    headers = {'Accept': 'application/json', 'Key': api_key}

    try:
        # Send the GET request to AbuseIPDB.
        response = requests.get(url, headers=headers, params=querystring, timeout=5)
        response.raise_for_status() # Raise an exception for HTTP errors (4xx or 5xx).
        data = response.json().get('data', {}) # Extract data from the JSON response.

        # Parse relevant information from the response.
        score = data.get('abuseConfidenceScore', 0)
        isp = data.get('isp', 'Unknown ISP')
        reports = data.get('totalReports', 0)
        return f"IP: {ip_address} | Abuse Score: {score}/100 | ISP: {isp} | Total Reports (90 days): {reports}"
    except requests.exceptions.RequestException as e:
        # Handle network or API request errors.
        return f"Failed to retrieve IP reputation for {ip_address}. Network/API Error: {str(e)}"

# Define another custom tool to query the NIST NVD for CVEs (Common Vulnerabilities and Exposures).
@tool
def lookup_cve(service_name: str) -> str:
    """
    Queries the NIST National Vulnerability Database (NVD) for recent CVEs related to a specific service or software.
    """
    url = "https://services.nvd.nist.gov/rest/json/cves/2.0"
    querystring = {'keywordSearch': service_name, 'resultsPerPage': 3}

    try:
        # Send the GET request to the NVD API.
        response = requests.get(url, params=querystring, timeout=10)
        response.raise_for_status()
        vulnerabilities = response.json().get('vulnerabilities', [])

        if not vulnerabilities:
            return f"No recent CVEs found in NVD for service: {service_name}."

        # Format the CVE summaries.
        cve_summaries = []
        for item in vulnerabilities:
            cve = item.get('cve', {})
            cve_id = cve.get('id', 'Unknown ID')
            desc_text = next((desc.get('value', '') for desc in cve.get('descriptions', []) if desc.get('lang') == 'en'), "No description.")
            cve_summaries.append(f"- {cve_id}: {desc_text}")
        return f"Top {len(cve_summaries)} CVEs related to '{service_name}':\n" + "\n".join(cve_summaries)
    except requests.exceptions.RequestException as e:
        return f"Failed to retrieve CVE data for {service_name}. API Error: {str(e)}"

# Function to initialize the Language Model (LLM) based on the specified provider.
def initialize_llm(provider="ollama"):
    if provider == "anthropic":
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model='claude-sonnet-4-5', temperature=0.2)
    elif provider == "ollama":
        # THE FIX 1: Use the dedicated `langchain_ollama` package for Ollama integration.
        from langchain_ollama import ChatOllama
        return ChatOllama(model="mistral", temperature=0.2)
    elif provider == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model="gpt-4o", temperature=0.2)
    raise ValueError(f"Unsupported LLM provider: {provider}")

# Determine the active LLM provider from environment variables, defaulting to 'ollama'.
active_provider = os.getenv("SOC_LLM_PROVIDER", "ollama")
# Initialize the LLM.
llm = initialize_llm(provider=active_provider)
# List of tools that the LLM agent can use.
tools = [lookup_ip_reputation, lookup_cve]

# Define the system prompt, which sets the persona and instructions for the LLM agent.
# This prompt guides the agent to act as a Tier-2 SOC analyst and produce a structured incident ticket.
SYSTEM_PROMPT = '''You are a Tier-2 SOC analyst assistant.
Produce a professional incident ticket in this exact structure:
1. Incident Summary (2-3 sentences: what happened, severity)
2. Attack Classification (type, confidence %)
3. Why flagged - Evidence (List the top features and their actual network metric 'true_value'. Explain why this specific value is suspicious in plain English. STRICT RULE: NEVER print the raw mathematical 'shap_value' floats.)
4. Immediate Containment Steps (numbered, actionable)
5. Investigation Queries (Write exact, executable Splunk SPL queries inside `code` blocks. Do not just describe the query.)
6. Escalation Recommendation (P1/P2/P3 with rationale)

CRITICAL INSTRUCTIONS:
- Do NOT invent evidence not in the SHAP bundle.
- You must OUTPUT ONLY the incident ticket.
- DO NOT output any Python code, scripts, or conversational filler. End your response immediately after the Escalation Recommendation.'''

# Create a React agent using LangGraph. The agent uses the LLM and provided tools
# to generate responses based on the system prompt.
# THE FIX 2: LangGraph V1.0+ uses 'prompt' directly instead of 'state_modifier'.
# Also, the import path for `create_react_agent` has moved, but `langgraph.prebuilt` still works.
agent_executor = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)


/tmp/ipykernel_1262/3422489459.py:113: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)


In [16]:
# Define a function to process a single network connection, determine if it's anomalous,
# and if so, generate a SOC incident ticket using the LLM agent.
def process_connection(raw_row, src_ip=None):
    # Convert the raw connection data into a DataFrame format.
    row_df = pd.DataFrame([raw_row])
    # Scale the raw data using the previously fitted scaler. This is required for model prediction.
    row_scaled = scaler.transform(row_df)[0]

    # Predict the attack type using the Random Forest model.
    prediction = rf.predict(row_scaled.reshape(1,-1))[0]
    # Get the anomaly score from the Isolation Forest model.
    iso_score  = iso.decision_function(row_scaled.reshape(1,-1))[0]

    # If the Random Forest predicts 'normal' (0) AND Isolation Forest considers it normal (score > 0),
    # then the connection is not flagged as anomalous, and no ticket is generated.
    if prediction == 0 and iso_score > 0:
        return 'Connection classified NORMAL - no ticket generated.'

    # Create a SHAP explanation bundle for the flagged connection.
    # THE FIX: Pass both the raw DataFrame (`row_df`) and the scaled array (`row_scaled`)
    # to `explain_connection` so that true feature values can be extracted for interpretation.
    shap_bundle = explain_connection(row_df, row_scaled, int(prediction))
    # Add Isolation Forest score and source IP to the SHAP bundle for context.
    shap_bundle['isolation_forest_score'] = float(iso_score)
    shap_bundle['source_ip'] = src_ip or 'unknown'

    # Construct the user message for the LLM agent.
    # This message includes the SHAP analysis bundle, source IP, and timestamp,
    # providing all necessary information for the agent to generate an incident ticket.
    user_message = f'''
    Generate a SOC incident ticket for the following flagged connection:

    SHAP Analysis Bundle:
    {json.dumps(shap_bundle, indent=2)}

    Source IP: {src_ip or 'not captured'}
    Detection timestamp: {pd.Timestamp.now().isoformat()}
    '''

    # Invoke the LangGraph agent with the constructed user message.
    # The agent will use its LLM and tools to process this information
    # and generate an incident ticket based on the `SYSTEM_PROMPT`.
    result = agent_executor.invoke({"messages": [("user", user_message)]})
    # Return the content of the last message from the agent, which is the generated incident ticket.
    return result["messages"][-1].content

# Demonstrate the process by running it on the first flagged test row.
# Find the index of the first test sample where Random Forest predicted an attack.
flagged_idx = (rf_preds != 0).nonzero()[0][0]
# Process this flagged connection using the `process_connection` function.
# Provide a dummy source IP for demonstration purposes.
ticket = process_connection(X_test.iloc[flagged_idx], src_ip='192.168.1.47')
# Print the generated incident ticket.
print(ticket)


 Incident Summary:
A potential Denial of Service (DoS) attack has been detected on the network with a high confidence level of 99.69%. The source IP is 192.168.1.47.

Attack Classification:
Type: DoS, Confidence: 99.69%

Why flagged - Evidence:
1. Count: An unusually high number of packets (229.0) were observed from the source IP within a short timeframe, which is indicative of a potential DoS attack.
2. dst_host_rerror_rate: The destination host experienced an error rate of 1.0, suggesting that there may have been multiple failed connection attempts or packet loss, which can be a sign of a DoS attack.
3. flag_SF: The source IP does not have the SF flag set (0.0), which is typically used to identify friendly hosts in some network protocols. In this case, the absence of the flag might indicate malicious activity.
4. logged_in: No active user sessions were found on the destination host (0.0), which could suggest that the attacker is attempting to overwhelm the system with traffic while a